Imports

In [13]:
import os
from mistralai import Mistral

mistral configuration

In [14]:
api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
model = "open-mistral-nemo"
client = Mistral(api_key=api_key)


Request to mistral

In [15]:
chat_response = client.chat.complete(
    model= model,
    messages = [
        {
            "role": "user",
            "content": "What is the best French cheese?",
        },
    ]
)
print(chat_response.choices[0].message.content)

Choosing the "best" French cheese can be quite subjective as it depends on personal taste, but France is indeed renowned for its diverse and high-quality cheeses. Here are a few iconic ones that are often praised for their unique flavors and textures:

1. **Brie de Meaux**: This is a soft, creamy cheese made from cow's milk. It has a rich, buttery flavor and a bloomy rind. It's often considered one of the finest Bries.

2. **Camembert**: Similar to Brie, Camembert is a soft, surface-ripened cheese with a strong aroma and a rich, creamy texture. It's known for its distinctive flavor and is often enjoyed at room temperature.

3. **Roquefort**: This is a blue cheese made from sheep's milk. It has a tangy, pungent flavor and a crumbly texture. It's often served with fruit or nuts to balance its strong taste.

4. **Comté**: A semi-hard cheese made from cow's milk, Comté has a complex, nutty flavor and a smooth, creamy texture. It's often used in cooking due to its excellent melting properti

This is the prompt given by Sebastian DATEV

Define the System Message

In [15]:
system_message_template = """You are an experienced ontology and knowledge engineer. Your task is to create competency questions based on unstructured documents that will be used in a later stage to create an OWL ontology. Together with the document, you are given a short purpose description for the ontology that must be created. 

Remember the definition and characteristics of competency questions: 
Competency questions (CQs) are specific questions that an ontology should be able to answer once it is complete. They help  define the scope and validate the design of the ontology. 
Key Aspects of Good Competency Questions: 
    1. Aligned with Ontology Purpose: Reflect the domain and purpose, addressing critical use cases. 
    2. Clear and Unambiguous: Use concise, natural language to avoid misunderstandings. 
    3. Specific and Testable: Focus on precise aspects of the domain and ensure they can be validated with data or reasoning. 
    4. Categorized: Include retrieval (e.g., 'What is X?'), reasoning (e.g., 'What applies if X?'), and consistency questions (e.g., 'Is X valid?').""" 

Define the user message template

In [ ]:
user_message_template = """You are given the raw content that has been crawled from an HTML document together with an abstract description of the context of the document. Also, you are provided with the purpose of the ontology that is to be created from this document and other similar ones.

Abstract description of the document contents: 
    {AbstractDescription} 

General purpose of the ontology:
    {OntologyPurpose}
    
Document:
    {Document}
    
Your task is to generate competency questions based on the input. 
Here are the acceptance criteria for your output: 
    1. The competency questions should be formulated in the language of the input document. If the input document is in German, competency questions should be too. 
    2. For every competency question, insert the original passages of the document where you derived the competency question from."""

Prompt template

In [ ]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_message_template), 
        ("user", user_message_template), 
    ]) 

Identified Competency questions

In [ ]:
Siehe json Datei 

Ontology builder prompt 

In [ ]:
"role": "system", 
"content": (
## Role 
"You are an expert ontology engineer. Your task is to extract ontology components from the provided software help documents." 

"Ontology components include a) **Classes** (Nouns or noun phrases representing objects or concepts),  b) **Properties** (Verbs representing relationships or attributes), and c) **Relationships** (Connections between classes via properties)." 

## General Instructions 
"Identify and extract classes, properties, and relationships in the form of subject, predicate, object triples." 
"Follow the guidelines below when extracting and naming the ontology elements: " 

## Few-Shot Examples  
    "<Example 1>" 
        "Sentence: 'Ein Arbeitnehmer hat eine Sozialversicherungsnummer'" 
        "Classes: [Arbeitnehmner, Sozialversicherungsnummer]" 
        "Properties: [hatIdentifikation]" 
        "Relationship: [(Arbeitnehmer)-(hatIdentifikation)-(Sozialversicherungsnummer)]" 

    "<Example 2>" 
        "Sentence: 'Arbeitnehmer haben nach Gesetz einen geregelten Mindestanspruch auf 24 Tage Urlaub im Jahr'" 
        "Classes: [Arbeitnehmer, Urlaub]" 
        "Properties: [hatMindestAnspruchAuf]" 
        "Relationship: [(Arbeitnehmer)-(hatMindestAnspruchAuf)-(Urlaub)]" 

## Modelling Guidelines 
    "- **Concept Identification**:" 
        "- Identify nouns and noun phrases as potential **Classes**." 
        "- Identify verbs and verb phrases as potential **Properties**." 
        "- Identify prepositions that establish relationships between nouns." 
    "- **Classes**:" 
        "- Represent concepts in the domain, not the words that denote these concepts." 
        "- Avoid creating classes for synonyms; use a single class for concepts with the same meaning." 
    "- **Properties**:" 
        "- Represent significant relationships or attributes between classes." 
        "- Should be meaningful and represent a significant connection." 
    "- **Relationships**:" 
        "- Establish connections between classes using properties." 
        "- Ensure relationships are meaningful within the domain context." 
    "- **General Principles**:" 
        "- There is no single correct way to model a domain; the best solution depends on the application." 
        "- Ontology development is an iterative process; refine as needed." 
        "- Avoid cycles in the class hierarchy." 
        "- Siblings in the hierarchy should be at the same level of generality." 

## Naming Conventions 
    "**General**:" 
        "- Do not add strings like 'class', 'domain', 'range', 'property', or 'slot' to names." 
        "- Use consistent naming throughout the ontology." 
    "**Classes**:" 
        "- Names are always capitalized." 
        "- Use nouns or compound nouns (e.g., Vertrag, Arbeit, Arbeitsvertrag)." 
        "- Use singular over plural (e.g., Arbeitsvertrag instead of Arbeitsverträge)." 
        "- Avoid abbreviations (e.g., Arbeitgeber instead of AG)." 
    "**Properties**:" 
        "- Names start with a lower-case letter." 
        "- Use verbs or verb phrases." 
        "- Can contain nouns in CamelCase starting with a verb (e.g., hatAnspruchAuf)." 
        "- Do not include spaces, commas, asterisks, or special characters." 

##Finally 
    "Ensure that you include empty lists for classes, properties, or relationships if none are found." 
    "Make sure all extracted components, i.e. classes, properties, and relationships as well as all descriptions are in German and translate where necessary." 
    "Include a class only if at least one relationship is found for a class. Verify this requirement." 
    "As a postprocessing step, replace in the @id part of the ontology for all components and ensure that no blanks remain by replacing them with a dash. For example: @id: https://datev.de/ontology/Rechtskreis Bundesland Berlin should be changed to @id : https://datev.de/ontology/Rechtskreis-Bundesland-Berlin" 
    "Check carefully for each class without any relationship, based on the name and the description of the class, if it can be merged with another class or if it is actually a relationship between on class and another. This is particularly relevant for classes that are named in the form of 'has something'." 
    "All ontology components must be in German language!" 
    "Terms like 'range' or 'domain' are never allowed!" 
)}, 

{"role": "user", "content": f"Extract ontology components from the following text:\n\n{document_text}"} 

Canonicalization prompt

In [ ]:
f"Given the following list of synonymous terms:\n{terms}\n" 
f"With the provided DATEV terminology and linked context across the documents:\n{synonyms_context}\n" 

"Your task is to identify and consolidate terms into a single, unified German term that best represents the entire cluster. " 
"The goal is to form comprehensive and meaningful clusters by evaluating the semantic connections and relationships between terms in the context of the documents.\n" 
"Guidelines:\n" 
    "- Prioritize terms that best capture the broader meaning or purpose of the cluster in the given context.\n" 
    "- Ensure that the selected term represents the largest possible cluster of related terms while preserving accuracy.\n" 
    "- Avoid creating overly specific clusters based on minor or isolated links; instead, focus on connections that span across multiple documents.\n" 
    "- Retain domain-specific terminology, especially if it's central to the cluster's meaning.\n" 
    "- If multiple terms are equally valid, prefer the term most widely recognized in the DATEV context.\n" 
    "- Avoid abbreviations unless they are standard in the domain.\n" 
"After selecting the unified term:\n" 
    "- Ensure that all duplicates are removed to maintain a clean and consistent ontology.\n" 
    "- Return ONLY the selected term as plain text, with no additional explanation or formatting.\n" 

Consolidation prompt

In [ ]:
"role": "system", 
"content":(
"You are an ontology engineer expert specializing in canonicalizing terms and identifying synonyms." 
"Given a term and a list of nearest neighbors, your task is to identify the most representative term " 
"(canonical term) for the set of terms and provide correct synonyms. " 
"For this task, only consider terms from the provided list of neighbors." 
"The canonical term should be the one that best represents the list in professional contexts, i.e. consider " 
"legal, administrative, business and technical similarity to find the canonical term." 
"Ensure that identified canonical terms are true synonyms, i.e., they are interchangeable or represent the same concept." 
"Do not include terms as synonyms that are merely related or similar. " 
"The canonical term appears before the colon (':') " 
"and the list of synonyms follows as an array of strings. " 
"If none of the neighbors are true synonyms satisfying the requirements, respond with an empty array for synonyms. " 
"All responses must be in German." 

Ontology evaluation using Competency questions prompt - kann besser gemacht warden 

In [ ]:
f"Given the following competency questions: {question}\n" 
f"Given also the following ontology: {self.ontology}\n" 

"Please check whether the competency question can be answered from the information given in the ontology.\n" 
"As a result return only yes or no!" 

Evaluation report

In [ ]:
{"Structural Consistency and Completeness": { 
"Class Count": 186, 
"Property Count": 171, 
"Relationship Count": 223, 
"Average Properties per Class": 0.9193548387096774, 
"Root Class Count": 56, 
"Hierarchy Depth": 5 
}, 
"Semantic Accuracy and Correctness": { 
"Has Cycles": false 
}, 
"Cohesion and Redundancy": { 
"Average Relationships per Class": 1.1989247311827957, 
"Redundant Classes": 0 
}, 
"Domain Coverage and Concept Density": { 
"Domain Coverage (%)": 8.536301651783319, 
"Concept Density": 13.324022346368714 
}, 
"Modularity and Extensibility": { 
"Number of Modules": 41, 
"Average Module Size": 145 
}, 
"Competency_based_eval": { 
"percentage_answered": 0.7440758293838863, 
"not answered competency questions": [ 
["Input/1033073.html", 
    "Welche Schritte sind erforderlich, um eine neue Kontenbeschriftung zu erfassen?"], 
["Input/1033073.html", 
    "Wie kann die Sachkontenlänge bei einem bestehenden Mandanten geändert werden?" ], 
["Input/1033073.html", 
    "Welche Schritte sind notwendig, um eine Mitarbeitergruppe anzulegen und zuzuordnen?"], 
["Input/9243155.html",
    "Welche Informationen müssen in den Mandantendaten erfasst werden?"], 
["Input/9243155.html", 
    "Wer legt die Einträge in der Liste Rechtsform fest, die in LODAS verwendet werden?"], 
["Input/1070718.html", 
    "Welche Voraussetzungen müssen erfüllt sein, um eine Arbeitsbescheinigung elektronisch zu übermitteln?"], 
["Input/1070718.html",
    "Welche Personengruppen sind von der Erstellung einer Arbeitsbescheinigung ausgeschlossen?"], 
["Input/9243600.html", 
    "Welche Schritte sind erforderlich, um Zusatzangaben zur Kündigung zu erfassen?"], 
["Input/9243600.html", 
    "Unter welchen Bedingungen sollte das Kontrollkästchen 'betriebsbedingte Kündigung mit Abfindungsangebot gem. § 1a KSchG' aktiviert werden?"], 
["Input/9222485.html",
    "Welche Angaben müssen erfasst werden, um einen neuen Mandanten anzulegen?"], 
["Input/9222485.html", 
    "Welche Informationen müssen für das Bundesland Berlin zusätzlich erfasst werden?"],
["Input/1007873.html", 
    "Wie wird die Verteilung der Festbezüge nach geleisteten Stunden durchgeführt?"], 
["Input/9245106.html", 
    "Welche Zeichen sind im Feld Straße zulässig?"], 
["Input/9245106.html",
    "Wie wird die ausländische Postleitzahl geprüft?"], 
["Input/9243601.html", 
    "Wie wird das Datum erfasst, an dem der Arbeitgeber die Kündigung ausgesprochen hätte, wenn der Arbeitnehmer nicht selbst gekündigt hätte?"], 
["Input/9245184.html", 
    "Welche Schritte sind erforderlich, um die Entlohnung eines Mitarbeiters in der Schnellerfassung zu erfassen?"], 
["Input/9219330.html", 
    "Welche Ausnahmen gibt es bei der Beitragspflicht in Bremen?"],
["Input/9219330.html",
    "Welche Arbeitnehmer sind im Saarland von der Beitragspflicht ausgeschlossen?"],
["Input/9219330.html", 
    "Wie hoch ist der Beitragshöchstbetrag im Saarland für 2023?"], 
["Input/9219330.html", 
    "Was passiert, wenn das Bundesland des Arbeitgebers abweicht?"], 
["Input/9274591.html", 
    "Welche Schritte sind erforderlich, um Angaben zur Befristung zu erfassen?"], 
["Input/9274591.html", 
    "Welche Informationen müssen im Feld 'Befristung Arbeitsvertrag bei Abschluss zum (TT.MM.JJJJ)' erfasst werden?"], 
["Input/9274591.html", 
    "Welche Daten müssen im Feld 'Abschluss Arbeitsvertrag am (TT.MM.JJJJ)' eingegeben werden?"], 
["Input/9274591.html", 
    "Welche Informationen sind im Feld 'Letzte Verlängerung des Arbeitsvertrages am (TT.MM.JJJJ)' erforderlich?"], 
["Input/9274591.html", 
    "Welche Informationen sind im Feld 'Letzte Verlängerung des Arbeitsvertrages bis (TT.MM.JJJJ)' erforderlich?"], 
["Input/1070191.html", 
    "Welche Auswertungen sind im Zahlstellen-Meldeverfahren vorgesehen?"], 
["Input/9245341.html", 
    "Welche Bedingungen müssen erfüllt sein, um einen offenen Beschäftigungszeitraum zu erfassen?"], 
["Input/9245341.html", 
    "Welche Informationen müssen im Feld 'Ersteintrittsdatum' eingegeben werden?"], 
["Input/9222342.html", 
    "Wie wird der Verzicht auf eine Abfindung erfasst?"], 
["Input/9219337.html", 
    "Welche Besonderheiten gibt es bei einem Austritt im Vorjahr?"], 
["Input/9219337.html",
    "Was passiert mit Arbeitnehmern, deren Austritt mehr als zwei Kalenderjahre zurückliegt?"], 
["Input/9219337.html", 
    "Wie wird das Feld 'Abw. Beginn Arbeitsverhältnis' verwendet?"], 
["Input/9219337.html", 
    "Was ist zu beachten, wenn das Austrittsdatum im Vormonat nicht erfasst wurde?"], 
["Input/9219337.html", 
    "Was ist zu tun, wenn ein neuer Beschäftigungszeitraum abgerechnet wurde, aber eine Korrektur des vorherigen Zeitraums erforderlich ist?"], 
["Input/9219337.html", 
    "Welche Bedingungen müssen erfüllt sein, damit eine Personalnummer abgerechnet werden kann?"], 
["Input/9221152.html", 
    "Wie kann man Abteilungen in den Mandantendaten anlegen?"], 
["Input/9221152.html", 
    "Wie kann man Abteilungen übergreifend ändern?"], 
["Input/1070573.html", 
    "Welche Bedingungen müssen erfüllt sein, damit negative Beiträge nicht an das Versorgungswerk übermittelt werden?"], 
["Input/1070573.html", 
    "Wie können individuelle DÜ-Zahlungstermine für Versorgungswerke festgelegt werden?"], 
["Input/9245340.html", 
    "Wie wird das Ersteintrittsdatum für AAG und Brutto/Netto-Formular verwendet?"], 
["Input/9245340.html", 
    "Wie viele Beschäftigungszeiträume dürfen pro Bearbeitungsmonat erfasst werden?"], 
["Input/9245340.html", 
    "Was muss getan werden, wenn ein neuer Beschäftigungszeitraum abgerechnet wurde und eine Nachberechnung auf vorherige Zeiträume erforderlich ist?"], 
["Input/9219339.html", 
    "Welche Gründe können bei Ungewissheit der Zahlung angegeben werden?"], 
["Input/9222474.html", 
    "Wie wird eine ausländische Bankverbindung erfasst?"], 
["Input/1070418.html", 
    "Was sind die Voraussetzungen für eine geringfügig entlohnte Beschäftigung bis zum 30.09.2022?"], 
["Input/1070418.html", 
    "Welche Regelungen gelten für die Bestandsschutzregelung bis zum 31.12.2023?"], 
["Input/1070418.html", 
    "Welche Lohnarten müssen für die Abrechnung einer geringfügigen Beschäftigung erfasst werden?"], 
["Input/9219333.html", 
    "Welche Dokumentnummern sind mit der Erfassung von Kündigungsangaben verbunden?"], 
["Input/9219333.html", 
    "Welche Dokumentnummer ist mit der Erfassung von Zahlungen bei Austritt verbunden?"], 
["Input/9219333.html", 
    "Welche Dokumentnummer ist mit der Übermittlung der Arbeitsbescheinigung verbunden?"], 
["Input/9244734.html", 
    "Welche Informationen können gedruckt werden, wenn man auf das Symbol 'Liste drucken' klickt?"], 
["Input/9244734.html", 
    "Wie kann der Datenpfad für die LODAS-Datenbank festgelegt werden?"], 
["Input/1021811.html", 
    "Wie wird die automatische Ermittlung der Brutto- und Netto-Werte durchgeführt?"], 
["Input/1021811.html", 
    "Welche Rolle spielt die ITSG bei der Ermittlung der Brutto- und Netto-Entgelte?"] 
]}} 